### VQWWVAE

In [ ]:
import numpy as np
import os
import pandas as pd
from torch.utils.data import DataLoader, TensorDataset
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from typing import List, TypeVar, Any
Tensor = TypeVar('torch.tensor')

is_norm = True
new_min, new_max = -1, 1

input_dim = 15
hidden_dim = [64,128]
seq_len = 150
embedding_dim = 128
num_embeddings = 512

In [ ]:
def data_to_model(data, sequence_length=120, step_size=25):
    data = data.float()
    sequences = []
    N = data.shape[0]
    
    for start_idx in range(0, N - sequence_length + 1, step_size):
        sequence = data[start_idx:start_idx + sequence_length]
        sequences.append(sequence)
    
    sequences = torch.stack(sequences)
    return sequences

def get_data(mode = '1-hover', phase = '2_5'):
    # 1-hover 2-forward 3-acc-x-axis-flag3-3000 5-motor03-flag4-085 6-motor03-flag3-200
    fmode = mode
    base_path = os.getcwd().split('\model')[0]

    sycn_raw_data = f'all_sycn_raw_data_mode{phase}.csv'
    sycn_err_data = f'all_sycn_err_data_mode{phase}.csv'
    sycn_sim_data = f'all_sycn_sim_data_mode{phase}.csv'

    sycn_raw_data_path = os.path.join(base_path, 'data', 'Dpro', fmode, sycn_raw_data)
    sycn_err_data_path = os.path.join(base_path, 'data', 'Dpro', fmode, sycn_err_data)
    sycn_sim_data_path = os.path.join(base_path, 'data', 'Dpro', fmode, sycn_sim_data)

    raw_data = pd.read_csv(sycn_raw_data_path).to_numpy()
    err_data = pd.read_csv(sycn_err_data_path).to_numpy()
    sim_data = pd.read_csv(sycn_sim_data_path).to_numpy()

    return raw_data, sim_data, err_data

def get_train_data(mode= ['1-hover', '2-forward', '1-hover'], phase = ['2_6', '2_5', '2_3']):
    fi_raw, fi_sim, fi_err = get_data(mode= mode[0], phase=phase[0])
    sim_data = np.vstack((fi_sim))
    raw_data = np.vstack((fi_raw))
    err_data = np.vstack((fi_err))
    for mo, ph in zip(mode[1:], phase[1:]):
        raw, sim, err = get_data(mode= mo, phase= ph)
        sim_data = np.vstack((sim_data, sim))
        raw_data = np.vstack((raw_data, raw))
        err_data = np.vstack((err_data, err))

    sim_data_split = np.split(sim_data, indices_or_sections=6, axis=1)
    raw_data_split = np.split(raw_data, indices_or_sections=6, axis=1)
    err_data_split = np.split(err_data, indices_or_sections=6, axis=1)

    sim_gyro, sim_acc, sim_mag, sim_pos, sim_vel, sim_eacc = sim_data_split[0], sim_data_split[1], sim_data_split[2], sim_data_split[3], sim_data_split[4], sim_data_split[5]
    raw_gyro, raw_acc, raw_mag, raw_pos, raw_vel, raw_eacc = raw_data_split[0], raw_data_split[1], raw_data_split[2], raw_data_split[3], raw_data_split[4], raw_data_split[5]
    err_gyro, err_acc, err_mag, err_pos, err_vel, err_eacc = err_data_split[0], err_data_split[1], err_data_split[2], err_data_split[3], err_data_split[4], err_data_split[5]

    sim_data_ = np.hstack((sim_acc, sim_gyro, sim_mag, sim_pos, sim_vel))
    raw_data_ = np.hstack((raw_acc, raw_gyro, raw_mag, raw_pos, raw_vel))
    err_data_ = np.hstack((err_acc, err_gyro, err_mag, err_pos, err_vel))

    return torch.tensor(sim_data_), torch.tensor(raw_data_), torch.tensor(err_data_)

def get_sensor_data(mode= ['1-hover', '2-forward', '1-hover'], phase = ['2_6', '2_5', '2_3'], label = 1, shuffle=False, ratio=1, sequence_length=80, step_size=1):
    sim, raw, err = get_train_data(mode, phase)

    if is_norm:
        sim_min_vals = torch.min(sim, dim=0)[0] 
        sim_max_vals = torch.max(sim, dim=0)[0]
        sim_normalized = ((sim - sim_min_vals) / (sim_max_vals - sim_min_vals)) * (new_max - new_min) + new_min

        raw_min_vals = torch.min(raw, dim=0)[0] 
        raw_max_vals = torch.max(raw, dim=0)[0]
        raw_normalized = ((raw - raw_min_vals) / (raw_max_vals - raw_min_vals)) * (new_max - new_min) + new_min

        err_min_vals = torch.min(err, dim=0)[0] 
        err_max_vals = torch.max(err, dim=0)[0]
        err_normalized = ((err - err_min_vals) / (err_max_vals - err_min_vals)) * (new_max - new_min) + new_min
    else:
        sim_min_vals, sim_max_vals, raw_min_vals, raw_max_vals, err_min_vals, err_max_vals = None, None, None, None, None, None
        sim_normalized, raw_normalized, err_normalized = sim, raw, err

    sim_2_model = data_to_model(data=sim_normalized, sequence_length=sequence_length, step_size=step_size)
    sim_2_model_label = torch.full((sim_2_model.size(0), 1), label, dtype=torch.long)

    raw_2_model = data_to_model(data=raw_normalized, sequence_length=sequence_length, step_size=step_size)
    raw_2_model_label = torch.full((raw_2_model.size(0), 1), label, dtype=torch.long)

    err_2_model = data_to_model(data=err_normalized, sequence_length=sequence_length, step_size=step_size)
    err_2_model_label = torch.full((err_2_model.size(0), 1), label, dtype=torch.long)

    if shuffle:
        torch.manual_seed(42)
        indices = torch.randperm(sim_2_model.size(0))

        sim_2_model = sim_2_model[indices]
        sim_2_model_label = sim_2_model_label[indices]
        raw_2_model = raw_2_model[indices]
        raw_2_model_label = raw_2_model_label[indices]
        err_2_model = err_2_model[indices]
        err_2_model_label = err_2_model_label[indices]

    sim_2_model = sim_2_model[:int(sim_2_model.size(0) * ratio)]
    sim_2_model_label = sim_2_model_label[:int(sim_2_model_label.size(0) * ratio)]
    raw_2_model = raw_2_model[:int(raw_2_model.size(0) * ratio)]
    raw_2_model_label = raw_2_model_label[:int(raw_2_model_label.size(0) * ratio)]
    err_2_model = err_2_model[:int(err_2_model.size(0) * ratio)]
    err_2_model_label = err_2_model_label[:int(err_2_model_label.size(0) * ratio)]

    return sim_2_model, sim_2_model_label, raw_2_model, raw_2_model_label, err_2_model, err_2_model_label, (raw_min_vals, raw_max_vals), (sim_min_vals, sim_max_vals), (err_min_vals, err_max_vals)

def normalize_data(x, data_min, data_max):
    return (x - data_min) / (data_max - data_min)

def denormalize_data(x, data_min, data_max):
    return ((x - new_min) / (new_max - new_min)) * (data_max - data_min) + data_min

# 1-hover 2-forward 3-acc-x-axis-flag3-3000 5-motor03-flag4-085 6-motor03-flag3-200
hover_2_model_sim, hover_2_model_sim_lables, hover_2_model_raw, hover_2_model_raw_lables, hover_2_model_err, hover_2_model_err_lables, (hover_raw_min_vals, hover_raw_max_vals), (hover_sim_min_vals, hover_sim_max_vals), (hover_err_min_vals, hover_err_max_vals) = get_sensor_data(mode= ['1-hover'], phase = ['2_6'], label = 1, shuffle=True, ratio=1, sequence_length=seq_len, step_size=4)
takeoff_2_model_sim, takeoff_2_model_sim_lables, takeoff_2_model_raw, takeoff_2_model_raw_lables, takeoff_2_model_err, takeoff_2_model_err_lables, (takeoff_raw_min_vals, takeoff_raw_max_vals), (takeoff_sim_min_vals, takeoff_sim_max_vals), (takeoff_err_min_vals, takeoff_err_max_vals) = get_sensor_data(mode= ['1-hover'], phase = ['2_3'], label = 2, shuffle=True, ratio=1, sequence_length=seq_len, step_size=4)

err_2_model = torch.cat((hover_2_model_err, takeoff_2_model_err), dim=0)
raw_2_model = torch.cat((hover_2_model_raw, takeoff_2_model_raw), dim=0)
sim_2_model = torch.cat((hover_2_model_sim, takeoff_2_model_sim), dim=0)
label_2_model = torch.cat((hover_2_model_raw_lables, takeoff_2_model_raw_lables), dim=0)

label_ana = {
    '1': (hover_raw_min_vals, hover_raw_max_vals, hover_sim_min_vals, hover_sim_max_vals, hover_err_min_vals, hover_err_max_vals),
    '2': (takeoff_raw_min_vals, takeoff_raw_max_vals, takeoff_sim_min_vals, takeoff_sim_max_vals, takeoff_err_min_vals, takeoff_err_max_vals)
}

print(f'hover_2_model_sim:{hover_2_model_sim.shape} takeoff_2_model_sim:{takeoff_2_model_sim.shape}')
print(f'err_2_model:{err_2_model.shape} label_2_model:{label_2_model.shape}')

dataset = TensorDataset(err_2_model, raw_2_model, sim_2_model, label_2_model)
batch_size = 4 
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last=True)
sh = next(iter(dataloader))[0].shape
print(f'dataloader Len: {len(dataloader)} dataloader Shape: {sh}')


In [ ]:
from abc import abstractmethod

class BaseVAE(nn.Module):
    
    def __init__(self) -> None:
        super(BaseVAE, self).__init__()

    def encode(self, input: Tensor) -> List[Tensor]:
        raise NotImplementedError

    def decode(self, input: Tensor) -> Any:
        raise NotImplementedError

    def sample(self, batch_size:int, current_device: int, **kwargs) -> Tensor:
        raise NotImplementedError

    def generate(self, x: Tensor, **kwargs) -> Tensor:
        raise NotImplementedError

    @abstractmethod
    def forward(self, *inputs: Tensor) -> Tensor:
        pass

    @abstractmethod
    def loss_function(self, *inputs: Any, **kwargs) -> Tensor:
        pass

In [ ]:
class VectorQuantizer(nn.Module):
    def __init__(self,
                 num_embeddings: int,  # K
                 embedding_dim: int,   # D
                 beta: float = 0.25):
        super(VectorQuantizer, self).__init__()
        self.K = num_embeddings
        self.D = embedding_dim
        self.beta = beta

        self.embedding = nn.Embedding(self.K, self.D)
        self.embedding.weight.data.uniform_(-1 / self.K, 1 / self.K)

    def forward(self, latents: Tensor) -> Tensor:
        latents = latents.permute(0, 2, 1)  # [B x N x D] -> [B x D x N] (adjusted for time-series)
        latents_shape = latents.shape
        flat_latents = latents.reshape(-1, self.D)  # [B*N x D]

        # Compute L2 distance between latents and embedding weights
        dist = torch.sum(flat_latents ** 2, dim=1, keepdim=True) + \
               torch.sum(self.embedding.weight ** 2, dim=1) - \
               2 * torch.matmul(flat_latents, self.embedding.weight.t())  # [B*N x K]

        # Get the encoding that has the min distance
        encoding_inds = torch.argmin(dist, dim=1).unsqueeze(1)  # [B*N, 1]

        # Convert to one-hot encodings
        device = latents.device
        encoding_one_hot = torch.zeros(encoding_inds.size(0), self.K, device=device)
        encoding_one_hot.scatter_(1, encoding_inds, 1)  # [B*N x K]

        # Quantize the latents
        quantized_latents = torch.matmul(encoding_one_hot, self.embedding.weight)  # [B*N, D]
        quantized_latents = quantized_latents.view(latents_shape)  # [B x D x N]

        # Compute the VQ Losses
        commitment_loss = F.mse_loss(quantized_latents.detach(), latents)
        embedding_loss = F.mse_loss(quantized_latents, latents.detach())

        vq_loss = commitment_loss * self.beta + embedding_loss 

        # Add the residue back to the latents
        quantized_latents = latents + (quantized_latents - latents).detach()

        return quantized_latents.permute(0, 2, 1), vq_loss  # [B x N x D] (adjusted back)
    
class VQVAE(BaseVAE):
    def __init__(self,
                 in_channels: int,  
                 embedding_dim: int,
                 num_embeddings: int,
                 hidden_dims: List = None,
                 latent_dim: int = 128,
                 beta: float = 0.25,
                 seq_len: int = 64,  
                 **kwargs) -> None:
        super(VQVAE, self).__init__()

        self.embedding_dim = embedding_dim
        self.num_embeddings = num_embeddings
        self.seq_len = seq_len
        self.beta = beta

        if hidden_dims is None:
            hidden_dims = [128, 256]

        self.lstm_layers = nn.ModuleList()
        input_dim = in_channels  
        for hidden_dim in hidden_dims:
            self.lstm_layers.append(nn.LSTM(input_size=input_dim, hidden_size=hidden_dim, batch_first=True))
            input_dim = hidden_dim  
        
        self.fc_mu = nn.Linear(hidden_dims[-1], latent_dim)
        self.fc_var = nn.Linear(hidden_dims[-1], latent_dim)

        self.vq_layer = VectorQuantizer(num_embeddings,
                                        embedding_dim,
                                        self.beta)

        self.lstm_decoders = nn.ModuleList()

        input_dim = hidden_dims[-1]
        for h_dim in reversed(hidden_dims):
            self.lstm_decoders.append(nn.LSTM(input_dim, h_dim, batch_first=True))
            # print(f'Decoder LSTM Dim:({input_dim, h_dim})')
            input_dim = h_dim

        self.output_lstm = nn.LSTM(input_size=hidden_dims[0], hidden_size=in_channels, batch_first=True)

    def encode(self, input: Tensor) -> Tensor:
        result = input
        for lstm in self.lstm_layers:
            result, _ = lstm(result)  

        mu = self.fc_mu(result)
        log_var = self.fc_var(result)
        z = self.reparameterize(mu, log_var)

        return z  

    def decode(self, z: Tensor) -> Tensor:
        result = z
        for lstm in self.lstm_decoders:
            result, _ = lstm(result) 
        result, _ = self.output_lstm(result)  
        return result
    
    def reparameterize(self, mu: Tensor, logvar: Tensor) -> Tensor:
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return eps * std + mu

    def forward(self, input: Tensor, **kwargs) -> Tensor:
        encoding = self.encode(input)
        quantized_inputs, vq_loss = self.vq_layer(encoding)
        return self.decode(quantized_inputs), vq_loss

    def loss_function(self,
                      *args,
                      **kwargs) -> dict:

        recons = args[0]  
        input = args[1]  
        vq_loss = args[2]  

        recons_loss = F.mse_loss(recons, input)


        mean_x = torch.mean(input, dim=(1)) 
        std_x = torch.std(input, dim=(1))   
        w_loss = self.wasserstein_loss_weighted(input, recons, mean_x, std_x)

        loss = recons_loss + vq_loss + w_loss
        return {'loss': loss,
                'Reconstruction_Loss': recons_loss,
                'VQ_Loss': vq_loss,
                'W_Loss': w_loss}

    def wasserstein_loss_weighted(self, x, recon_x, mean_x, std_x)
        mse_recon_loss = F.mse_loss(recon_x, x)
        loss_i = mean_x - 0 + std_x - 1 + mse_recon_loss
        weights = F.softmax(loss_i, dim=0)
        weighted_loss = weights * loss_i
        return torch.mean(weighted_loss)
    

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
vae = VQVAE(in_channels=input_dim, hidden_dims=hidden_dim, embedding_dim=embedding_dim, num_embeddings=num_embeddings).to(device)
optimizer = optim.Adam(vae.parameters(), lr=1e-3)
print(vae)

In [ ]:
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)

def train_vae(dataloader, vae, num_epochs):
    vae.train()
    epoch_losses = []
    fig_cnt= 1
    plt.figure(figsize=(16, 4))
    for epoch in range(num_epochs):
        total_loss = 0.0
        total_reconstruction_loss = 0
        total_vq_loss = 0
        total_w_loss = 0
        for err, _, _, _ in dataloader:
            optimizer.zero_grad()

            input_data = err  
            reconstructed, vq_loss = vae(input_data.to(device))
            loss_dict = vae.loss_function(reconstructed, input_data.to(device), vq_loss)
            loss = loss_dict['loss']
            reconstruction_loss = loss_dict['Reconstruction_Loss']
            vq_loss = loss_dict['VQ_Loss']
            w_loss = loss_dict['W_Loss']
            
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            total_reconstruction_loss += reconstruction_loss.item()
            total_vq_loss += vq_loss.item()
            total_w_loss += w_loss.item()
        

        epoch_losses.append(total_loss / len(dataloader))
        print(f"Epoch [{epoch+1}/{num_epochs}], "
              f"Total loss: {total_loss / len(dataloader):.4f}, "
              f"Recon loss: {total_reconstruction_loss / len(dataloader):.4f}, "
              f"VQ loss: {total_vq_loss / len(dataloader):.4f}, "
              f"W loss: {total_w_loss / len(dataloader):.4f}")

        if epoch % 50 == 0:
            data = reconstructed[0].cpu().detach().numpy()  
            tsne = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=1000)
            data_2d = tsne.fit_transform(data)

            plt.subplot(1, num_epochs // 50, fig_cnt)
            plt.scatter(data_2d[:, 0], data_2d[:, 1], s=10, alpha=0.7, cmap='viridis')
            plt.title("t-SNE Visualization of Data", fontsize=10)
            plt.xlabel("t-SNE Dimension 1")
            plt.ylabel("t-SNE Dimension 2")
            plt.grid(True)
            fig_cnt += 1
    
    plt.figure(figsize=(4, 4))
    plt.plot(epoch_losses, label='Training Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('VAE Training Loss Curve')
    plt.legend()
    plt.show()

num_epochs = 250
train_vae(dataloader, vae, num_epochs)

In [ ]:
def test_vae(vae, test_loader, device):
    vae.eval()
    original_data = []
    reconstructed_data = []

    with torch.no_grad():
        for err, raw, sim, label in test_loader:
            err = err.to(device)
            label = label.to(device)
            recon_err, vq_loss = vae(err)
            recon_err = recon_err.cpu()
            label = label.cpu()
            err = err.cpu()
            
            if is_norm:
                raw_min_vals = label_ana[str(label[0][0].numpy())][0]
                raw_max_vals = label_ana[str(label[0][0].numpy())][1]
                sim_min_vals = label_ana[str(label[0][0].numpy())][2]
                sim_max_vals = label_ana[str(label[0][0].numpy())][3]
                err_min_vals = label_ana[str(label[0][0].numpy())][4]
                err_max_vals = label_ana[str(label[0][0].numpy())][5]

                err_denorm = denormalize_data(err, err_min_vals, err_max_vals)
                recon_x_denorm = denormalize_data(recon_err, err_min_vals, err_max_vals) 

                raw_denorm = denormalize_data(raw, raw_min_vals, raw_max_vals)
                sim_denorm = denormalize_data(sim, sim_min_vals, sim_max_vals)

                sim_recon_denorm = sim_denorm - recon_x_denorm
            else:
                err_denorm = err
                recon_x_denorm = recon_err
                raw_denorm = raw
                sim_denorm = sim
                sim_recon_denorm = sim_denorm - recon_x_denorm
             
            original_data.append(raw_denorm.cpu())
            reconstructed_data.append(sim_recon_denorm.cpu())

    print(f'original_data:{len(original_data)}')
    original_data = torch.cat(original_data, dim=0)
    print(f'original_data:{original_data.shape}')
    reconstructed_data = torch.cat(reconstructed_data, dim=0)

    return original_data, reconstructed_data

original_data, reconstructed_data = test_vae(vae, dataloader, device)

In [ ]:

look_dim = 8

def plot_reconstruction(original_data, reconstructed_data, dim=look_dim):
    plt.figure(figsize=(12, 6))
    seq_len = original_data.size(1)
    
    for i in range(4):
        plt.subplot(2, 2, i + 1)
        plt.plot(range(seq_len), original_data[i][:,dim].numpy(), label="Original")
        plt.plot(range(seq_len), reconstructed_data[i][:,dim].numpy(), label="Reconstructed")
        plt.legend()
        plt.title(f"Sample {i+1}")
    
    plt.tight_layout()
    plt.show()

plot_reconstruction(original_data, reconstructed_data)

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

def plot_recon_data(plot_data=[takeoff_2_model_sim, takeoff_2_model_raw, takeoff_2_model_err], plot_batch=5, label=1, model=vae, is_norm=is_norm):
    _sim = plot_data[0][plot_batch]
    _raw = plot_data[1][plot_batch]
    _err = plot_data[2][plot_batch]

    err = _err.unsqueeze(0)

    model.eval()
    recon_err, _ = model(err.to(device))
    recon_err = recon_err.squeeze(0)
    recon_err = recon_err.cpu()

    if is_norm:
        raw_min_vals = label_ana[str(label)][0]
        raw_max_vals = label_ana[str(label)][1]
        sim_min_vals = label_ana[str(label)][2]
        sim_max_vals = label_ana[str(label)][3]
        err_min_vals = label_ana[str(label)][4]
        err_max_vals = label_ana[str(label)][5]

        _err = denormalize_data(_err, err_min_vals, err_max_vals)
        _sim = denormalize_data(_sim, sim_min_vals, sim_max_vals)
        _raw = denormalize_data(_raw, raw_min_vals, raw_max_vals)
        
        recon_x_denorm = denormalize_data(recon_err, err_min_vals, err_max_vals) 
        sim_recon_denorm = _sim - recon_x_denorm
    else:
        recon_x_denorm = recon_err
        sim_recon_denorm = _sim - recon_x_denorm
    
    _err = _err.detach().numpy()
    _sim = _sim.detach().numpy()
    _raw = _raw.detach().numpy()
    recon_x_denorm = recon_x_denorm.detach().numpy()
    sim_recon_denorm = sim_recon_denorm.detach().numpy()
    
    plt.figure(figsize=(16, 16))
    for i in range(input_dim):
        raw_data = _raw[:, i]
        sim_data = _sim[:, i]
        err_data = _err[:, i]
        recon_err_data = recon_x_denorm[:, i]
        sim_recon_data = sim_recon_denorm[:, i]
        plt.subplot(9, 2, i+1)
        plt.plot(raw_data, label='raw')
        plt.plot(sim_recon_data, label='recon_sim')
        # plt.plot(sim_recon_data, linestyle='--', label='recon')
        plt.title(f'Dim:{i}')
        plt.legend()
    
    rmse_list = []
    mae_list = []
    r2_list = []

    original_batch = _raw  
    reconstructed_batch = sim_recon_denorm  
    
    mse_batch = mean_squared_error(original_batch, reconstructed_batch)
    rmse_batch = np.sqrt(mse_batch)
    mae_batch = mean_absolute_error(original_batch, reconstructed_batch)
    r2_batch = r2_score(original_batch, reconstructed_batch)
    
    rmse_list.append(rmse_batch)
    mae_list.append(mae_batch)
    r2_list.append(r2_batch)

    avg_rmse = np.mean(rmse_list)
    avg_mae = np.mean(mae_list)
    avg_r2 = np.mean(r2_list)

    print(f"Reconstruction Errors (averaged over batches):")
    print(f"Average RMSE: {avg_rmse:.4f}")
    print(f"Average MAE: {avg_mae:.4f}")
    print(f"Average R²: {avg_r2:.4f}")

# plot_recon_data(plot_data=[takeoff_2_model_sim, takeoff_2_model_raw, takeoff_2_model_err], plot_batch=4, label=2, model=vae, is_norm=is_norm)
plot_recon_data(plot_data=[hover_2_model_sim, hover_2_model_raw, hover_2_model_err], plot_batch=1, label=1, model=vae, is_norm=is_norm)

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

vae.eval()  

reconstructions = []
originals = []

with torch.no_grad():
    for err, raw, sim, label in dataloader:
        err = err.to(device)
        label = label.to(device)
        recon_err, vq_loss = vae(err)
        label = label.cpu()
        recon_err = recon_err.cpu()
        err = err.cpu()
        
        if is_norm:
            raw_min_vals = label_ana[str(label[0][0].numpy())][0]
            raw_max_vals = label_ana[str(label[0][0].numpy())][1]
            sim_min_vals = label_ana[str(label[0][0].numpy())][2]
            sim_max_vals = label_ana[str(label[0][0].numpy())][3]
            err_min_vals = label_ana[str(label[0][0].numpy())][4]
            err_max_vals = label_ana[str(label[0][0].numpy())][5]

            err_denorm = denormalize_data(err, err_min_vals, err_max_vals)
            recon_x_denorm = denormalize_data(recon_err, err_min_vals, err_max_vals) 

            raw_denorm = denormalize_data(raw, raw_min_vals, raw_max_vals)
            sim_denorm = denormalize_data(sim, sim_min_vals, sim_max_vals)

            sim_recon_denorm = sim_denorm - recon_x_denorm
        else:
            err_denorm = err
            recon_x_denorm = recon_err
            raw_denorm = raw
            sim_denorm = sim
            sim_recon_denorm = sim_denorm - recon_x_denorm

        originals.append(raw_denorm.cpu())
        reconstructions.append(sim_recon_denorm.cpu())

reconstructed_data = torch.cat(reconstructions, dim=0)
original_data = torch.cat(originals, dim=0)

original_data_np = original_data.numpy()
reconstructed_data_np = reconstructed_data.numpy()

rmse_list = []
mae_list = []
r2_list = []

for i in range(original_data_np.shape[0]):
    original_batch = original_data_np[i]  # shape: (1000, 18)
    reconstructed_batch = reconstructed_data_np[i]  # shape: (1000, 18)
    
    mse_batch = mean_squared_error(original_batch, reconstructed_batch)
    rmse_batch = np.sqrt(mse_batch)
    mae_batch = mean_absolute_error(original_batch, reconstructed_batch)
    r2_batch = r2_score(original_batch, reconstructed_batch)
    
    rmse_list.append(rmse_batch)
    mae_list.append(mae_batch)
    r2_list.append(r2_batch)

avg_rmse = np.mean(rmse_list)
avg_mae = np.mean(mae_list)
avg_r2 = np.mean(r2_list)

print(f"Reconstruction Errors (averaged over batches):")
print(f"Average RMSE: {avg_rmse:.4f}")
print(f"Average MAE: {avg_mae:.4f}")
print(f"Average R²: {avg_r2:.4f}")


In [ ]:
import shutil

ReSaved = True

mode_name = 'vq_ww_vae.pth'
folder_name = mode_name.split('.')[0]
folder_path = os.path.join(os.getcwd(), folder_name)
path = os.path.join(os.getcwd(), folder_name, mode_name)

if not os.path.exists(folder_path):
    os.makedirs(folder_path, exist_ok=True)

torch.save(vae, path)
print(f'Model Re-Saved to {path}')